# 02 · Pipeline & Model — STFT, augmentation, and SingNet-C1

The engineering stage: the differentiable STFT front end, the chunk + augmentation pipeline, the mask target, and a walkthrough of the fixed **SingNet-C1** U-Net with its exact parameter table. Load-bearing logic lives in `singnet/`; this notebook orchestrates and visualises.

**No cell has been executed.** The CPU demos below *are* runnable when you choose to (synthetic signals, seconds); the smoke-overfit at the end is GPU RUN LATER.

### State of this notebook
- **Contract:** [`../MASTER_PLAN.md`](../MASTER_PLAN.md) §5 (model), §6 (losses), §7.1 (training), §14.
- **Theory:** [`../THEORY.md`](../THEORY.md) §1 (STFT/COLA), §5 (param count).
- **Tests mirror every claim here:** `tests/test_stft.py`, `tests/test_model.py`, `tests/test_augment.py`.

In [ ]:
# === Colab bootstrap — RUN THIS LATER (Colab only) ==========================
# ⚠️ RUN THIS LATER · ~2 min · CPU. Installs the pinned env and mounts Drive.
# Skip locally if you already `pip install -r requirements.txt`.
#
# !git clone https://github.com/SeanSalvador2/vocal-separation-research.git
# %cd vocal-separation-research
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# import os; os.environ["SHARD_ROOT"] = "/content/drive/MyDrive/musdb_shards"
print("Bootstrap cell — run on Colab only (see comments). No-op here.")


## 1 · STFT / iSTFT round-trip
The front end is `n_fft=4096`, `hop=1024`, Hann, `center=True` (THEORY §1). Perfect reconstruction holds because the squared Hann window tiles to a constant at 75 % overlap (the NOLA proof in THEORY §1.2). The demo below reconstructs a synthetic signal and reports the error in dB — **expect < −60 dB** (the unit test measures ≈ −137 dB). Runs on CPU in seconds.

In [ ]:
import torch
from singnet.audio import STFT
stft = STFT()
x = 0.1 * torch.randn(1, 264600)          # a 6 s mono chunk
x_hat = stft.inverse(stft.transform(x), length=x.shape[-1])
err_db = 10 * torch.log10(((x - x_hat) ** 2).mean() / (x ** 2).mean())
print(f'round-trip error: {float(err_db):.1f} dB  (want < -60 dB)')

## 2 · Chunking & augmentation gallery
The dataset samples a uniform-random 6 s chunk, mono-mixes down, and applies (in order, §4.5) cross-track **remix** → per-source **gain** U(0.25, 1.25) → **sign flip** p=0.5. The stream is a pure function of `(seed, index)` and never of the loss arm — the controlled-comparison invariant. The demo uses a synthetic in-memory store so it runs on CPU with no dataset; swap in `WavShardStore($SHARD_ROOT)` for the real thing.

In [ ]:
import numpy as np, pandas as pd
from singnet.data import InMemoryStore, MusdbChunks, Manifest
sr = 44100; rng = np.random.default_rng(0)
tracks = {f'train_{i}': {'vocals': 0.2*np.sin(2*np.pi*(200+30*i)*np.arange(sr*8)/sr).astype('float32'),
                         'accompaniment': 0.2*rng.standard_normal(sr*8).astype('float32')}
          for i in range(4)}
store = InMemoryStore(tracks, sr)
mani = Manifest(pd.DataFrame({'track': list(tracks), 'split': ['train']*4}))
ds = MusdbChunks(store, mani, 'train', seed=0, length=64)
item = ds[0]
print('chunk samples:', tuple(item['mixture'].shape))
print('mixture == vocals + accompaniment:',
      bool(torch.allclose(item['mixture'], item['vocals'] + item['accompaniment'], atol=1e-6)))
print('deterministic (ds[0] == ds[0]):', bool(torch.equal(ds[0]['mixture'], ds[0]['mixture'])))

*Figure to add when you run it:* overlay the raw vs augmented waveform and their spectrograms for one chunk — you should see the gain/sign changes in the waveform and an unchanged time-frequency *structure* (augmentation is amplitude/polarity, not spectral warping).

## 3 · Mask target (oracle IRM)
The learning target is bounded by the **oracle IRM** $M=|S|/(|S|+|A|+\varepsilon)$ (THEORY §2). Visualising it on one chunk shows what a perfect $[0,1]$ mask looks like — bright where vocals dominate, dark in accompaniment-only regions — and previews the ceiling the model chases.

In [ ]:
from singnet.audio import STFT, analyze_chunk
from singnet.eval import oracle_irm
voc = torch.from_numpy(store.load_sources('train_0')['vocals'][:264600]).float()[None]
acc = torch.from_numpy(store.load_sources('train_1')['accompaniment'][:264600]).float()[None]
_, voc_mag = analyze_chunk(STFT(), voc)
_, acc_mag = analyze_chunk(STFT(), acc)
irm = oracle_irm(voc_mag, acc_mag)
print('IRM shape:', tuple(irm.shape), '| range [', float(irm.min()), ',', float(irm.max()), ']')
# plt.imshow(irm[0].numpy(), origin='lower', aspect='auto')  # add when running

## 4 · SingNet-C1 architecture & parameter table
Five stride-2 encoder blocks (Conv5×5→BN→LeakyReLU) to a 512×64×8 bottleneck, a mirrored transposed-conv decoder with skips and Dropout2d(0.5) on the first three decoder blocks, then a 1×1 head + sigmoid. Featurization is $\log(1+|X|)$ standardized per chunk (THEORY §5).

| Block | params | · | Block | params |
|---|---:|---|---|---:|
| enc1 | 896 | · | dec1 | 3,277,568 |
| enc2 | 51,392 | · | dec2 | 1,638,784 |
| enc3 | 205,184 | · | dec3 | 409,792 |
| enc4 | 819,968 | · | dec4 | 102,496 |
| enc5 | 3,278,336 | · | dec5 | 51,296 |
| head | 33 | · | **total** | **9,835,745** |

The cell below rebuilds the model and asserts that exact count — the number in THEORY §5 and `tests/test_model.py` are the same number.

In [ ]:
from singnet.models import SingNetC1
from singnet.audio import N_MASK_BINS, N_FRAMES
model = SingNetC1()
assert model.num_parameters == 9_835_745, model.num_parameters
with torch.no_grad():
    mask = model(torch.rand(1, N_MASK_BINS, N_FRAMES))
print('params:', f'{model.num_parameters:,}')
print('mask shape:', tuple(mask.shape), '| range [0,1]:', float(mask.min()), float(mask.max()))

### Gradient-flow smoke check (CPU, seconds)
One forward+backward confirms gradients reach every parameter — the cheapest guard that the graph (including the differentiable iSTFT used by `sisdr`/`l1mrstft`) is wired correctly.

In [ ]:
model.zero_grad()
model(torch.rand(1, N_MASK_BINS, N_FRAMES)).sum().backward()
n_with_grad = sum(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
print(f'{n_with_grad} / {len(list(model.parameters()))} parameter tensors got finite gradients')

## 5 · Smoke-overfit gate (G1)
> ⚠️ **RUN THIS LATER** — overfit one 6 s chunk + a 5-track mini-run  ·  _< 1 GPU-h · T4_

**PASS criteria (MASTER_PLAN §8 step 2 / gate G1):** single-chunk train SI-SDR **> +20 dB within 2k steps**, *and* a 5-track mini-run beats the do-nothing floor on val-mini. Also record measured **steps/s** here to apply the §3.2 budget-scaling rule before spending on the sweep. Do **not** start the sweep until this passes.

In [ ]:
# ⚠️ RUN THIS LATER (GPU). Proof-it-trains before any sweep spend.
# !python -m singnet.train --config 01-loss-function-study/configs/smoke_overfit.yaml
print('G1 smoke test is RUN LATER — needs a GPU and the decoded data.')